# 06 · Context-ablation timing: instruction flag on/off

Measures how long **one** context-ablation pass takes for a single QA, and
whether folding the generation system prompt into the ablation set (the
`instruction` flag) changes the wall-clock time.

**Design choices:**
- **In-process via `attribute_one`, not the HTTP API.** `POST /attribute` is
  hardwired to a `CachingProvider` and never passes `instruction`, so it can
  satisfy neither requirement here. We call `attribute_one()` (a drop-in for the
  engine `attribute_by_sentence`, in `tools/attribute_one.py`) directly instead —
  it's also the sandbox where ablation concurrency will be prototyped later.
- **No cache.** We use a bare `OpenAICompatProvider` (never wrapped in
  `CachingProvider`), so every ablation call hits NIM fresh. Timing is real,
  not a replay of `./.cache/logprobs.sqlite`.

Run order: setup → pick QA → knobs → **Cell A (no instruction)** → **Cell B
(with instruction)** → compare.

## Setup — non-caching provider

In [ ]:
import time

from dare.config import Settings
from dare.providers import OpenAICompatProvider
from dare.schema import load_corpus
from dare.prompts import SYNTHESIS_SYSTEM

# attribute_one is a drop-in for dare.attribution.attribute_by_sentence (same args,
# same {response, whole, units} return). It delegates to the engine today; it's the
# sandbox where ablation concurrency gets prototyped later, so the cells below don't
# change when that lands. See tools/attribute_one.py.
from tools.attribute_one import attribute_one

settings = Settings.from_env()  # loads repo-root .env regardless of cwd

def fresh_provider():
    """A bare provider with NO caching wrapper — every score_response() call goes
    to NIM. A fresh instance per run so nothing (even in-memory) carries over."""
    return OpenAICompatProvider(settings)

print(f"model      : {settings.model}")
print(f"llm_url    : {settings.llm_url}")
print(f"call delay : {getattr(settings, 'api_call_delay', '?')}s between ablation calls (applies equally to both runs)")
print("caching    : OFF (bare OpenAICompatProvider)")

## Pick which QA to run

Set `QA_ID` to a record id, or leave it `None` and use `QA_INDEX`.

In [ ]:
CORPUS = "../data/raw_responses.jsonl"
QA_ID = None      # e.g. "qa-luongvh-0001"; if None, falls back to QA_INDEX
QA_INDEX = 0      # used only when QA_ID is None

records = load_corpus(CORPUS)

if QA_ID is not None:
    rec = next(r for r in records if r.id == QA_ID)
else:
    rec = records[QA_INDEX]

assert rec.attributable, f"record {rec.id!r} has no answer/sources to attribute"

print(f"id      : {rec.id}")
print(f"chunks  : {len(rec.chunks)}")
print(f"query   : {rec.query}")
print(f"answer  : {rec.answer[:300]}{'...' if len(rec.answer) > 300 else ''}")
print(f"\n(pick another: set QA_ID, or QA_INDEX in 0..{len(records) - 1})")

## Knobs

`NUM_ABLATIONS` drives the API-call count (dominant cost). `INSTRUCTION` is the
text folded in for Cell B — defaults to the real generation system prompt.

In [ ]:
NUM_ABLATIONS = 32          # ~this many sequential NIM calls per run
INSTRUCTION = SYNTHESIS_SYSTEM   # edit to test any instruction text

print(f"NUM_ABLATIONS = {NUM_ABLATIONS}")
print(f"INSTRUCTION   = {INSTRUCTION[:120]}...")

## Cell A — ablation WITHOUT the instruction flag

In [ ]:
t0 = time.perf_counter()
result_no_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    # instruction omitted -> context-only ablation
)
elapsed_no_instr = time.perf_counter() - t0

print(f"WITHOUT instruction: {elapsed_no_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_no_instr['units'])} units)")

## Cell B — ablation WITH the instruction flag

Same QA, same ablation count; the instruction is prepended into the context as
its own lane, so the ablation prompt is slightly longer per call.

In [ ]:
t0 = time.perf_counter()
result_with_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    instruction=INSTRUCTION,     # folded in -> origin='instruction' lane
)
elapsed_with_instr = time.perf_counter() - t0

print(f"WITH instruction   : {elapsed_with_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_with_instr['units'])} units)")

## Compare

In [ ]:
delta = elapsed_with_instr - elapsed_no_instr
pct = (delta / elapsed_no_instr * 100) if elapsed_no_instr else float('nan')

print(f"QA            : {rec.id}")
print(f"ablations     : {NUM_ABLATIONS}   chunks: {len(rec.chunks)}   cache: OFF")
print(f"without instr : {elapsed_no_instr:7.2f}s")
print(f"with instr    : {elapsed_with_instr:7.2f}s")
print(f"delta         : {delta:+7.2f}s  ({pct:+.1f}%)")